# Inspección legible de traces de M3

Este notebook lee archivos `results/*.jsonl`, selecciona episodios y muestra la traza paso a paso.

## `world`, `goal` y `check_goal`

En `build_agent`, `config["world"]` es la instancia mutable del mundo que las herramientas modifican. `config["goal"]` es la especificación del objetivo del escenario, por ejemplo `{"type": "item_open", "item": "puerta_principal"}`. Cuando ambos están disponibles, el agente puede consultar `mia_world.check_goal(world, goal)` para saber si el estado real ya cumple el objetivo.

La condición de ambos campos es deliberada: sin `world` no hay estado que inspeccionar; sin `goal` no hay qué verificar. Por eso el gate se activa únicamente en modo mundo y con un objetivo conocido. El API actual del repositorio recibe dos argumentos, `world` y `goal` (no un tercer `state`).

Los JSONL actuales guardan `goal` y `final_state`, pero no serializan la instancia viva `World` ni todos los mensajes crudos enviados al LLM. Por eso este notebook puede inspeccionar el trace persistido, pero no reconstruir literalmente cada prompt interno del proveedor.

In [77]:
from pathlib import Path
import json
from typing import Any, Iterable

WORKSPACE = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = WORKSPACE / "results"
DEFAULT_RESULTS_PATH = RESULTS_DIR / "qwen-easy-medium.jsonl" # qwen-study
RESULTS_PATH = DEFAULT_RESULTS_PATH
CASE_INDEX = 0
MAX_TEXT = 1200

files = sorted(RESULTS_DIR.glob("*.jsonl"))
print(f"Directorio: {RESULTS_DIR}")
print("Archivos disponibles:")
for index, path in enumerate(files):
    print(f"  [{index}] {path.name}")


Directorio: /home/felipe/git/udesa/tp_mia_agentes_2026/results
Archivos disponibles:
  [0] easy-medium-all-experiments.jsonl
  [1] exp-e.jsonl
  [2] qwen-easy-medium.jsonl
  [3] qwen-study.jsonl
  [4] smoke-easy-repair.jsonl
  [5] smoke-easy.jsonl
  [6] validate.jsonl


## 1 y 2. Carga robusta de JSONL

Cada línea no vacía representa un episodio, salvo la línea `_meta`, que contiene metadatos de la corrida. Los errores de parseo se reportan con archivo y número de línea; los campos opcionales no hacen fallar la carga.

In [78]:
def load_jsonl(path: str | Path) -> tuple[dict[str, Any], list[dict[str, Any]], list[str]]:
    path = Path(path)
    meta: dict[str, Any] = {}
    records: list[dict[str, Any]] = []
    errors: list[str] = []
    with path.open(encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(handle, 1):
            if not raw_line.strip():
                continue
            try:
                item = json.loads(raw_line)
            except json.JSONDecodeError as exc:
                errors.append(f"{path.name}:{line_number}: JSON inválido: {exc.msg}")
                continue
            if not isinstance(item, dict):
                errors.append(f"{path.name}:{line_number}: se esperaba un objeto JSON")
                continue
            if "_meta" in item:
                meta = item["_meta"] if isinstance(item["_meta"], dict) else {}
            else:
                if "trace" not in item:
                    errors.append(f"{path.name}:{line_number}: falta 'trace' (se conserva el registro)")
                records.append(item)
    return meta, records, errors

meta, records, load_errors = load_jsonl(RESULTS_PATH)
print(f"{len(records)} episodios cargados; {len(load_errors)} advertencias/errores")
for error in load_errors:
    print("!", error)


77 episodios cargados; 0 advertencias/errores


In [79]:
def check_configured_goal(config: dict[str, Any] | None) -> tuple[bool | None, str]:
    """Ejemplo defensivo: solo llama check_goal cuando hay world y goal."""
    config = config or {}
    world = config.get("world")
    goal = config.get("goal")
    if world is None or goal is None or goal == {} or goal == "":
        return None, "check_goal omitido por configuración incompleta"
    from mia_world import check_goal
    return check_goal(world, goal)[0], "check_goal ejecutado"

print(check_configured_goal({"world": None, "goal": {"type": "item_open"}})[1])
example_world = None
example_goal = {"type": "item_open", "item": "puerta_principal"}
print(check_configured_goal({"world": example_world, "goal": example_goal})[1])
print("Para ejecutar check_goal se necesita una instancia real de World, no un registro JSONL.")


check_goal omitido por configuración incompleta
check_goal omitido por configuración incompleta
Para ejecutar check_goal se necesita una instancia real de World, no un registro JSONL.


## 4. Normalizar roles y trace

Un trace de `eval/` guarda principalmente eventos de herramientas y observaciones. Para que la lectura sea clara, cada evento se desdobla en dos roles sintéticos: `assistant/tool_call` (la acción inferida a partir de `tool` y `args`) y `tool/observation` (el output real de la herramienta). También se usan `system` (instrucciones iniciales), `user` (pedido), `assistant` (texto o decisión) y `observation` (resultado).

Importante: los JSONL actuales no guardan todos los mensajes crudos enviados al LLM. Por eso `assistant/tool_call` no es el mensaje original del proveedor: es una reconstrucción fiel de la llamada registrada. El prompt inicial y el razonamiento textual solo se muestran si fueron persistidos en algún campo del registro.

In [80]:
ROLE_MAP = {
    "system": "system", "instruction": "system", "prompt": "system",
    "user": "user", "human": "user",
    "assistant": "assistant", "model": "assistant", "llm": "assistant",
    "tool": "tool", "function": "tool",
    "observation": "observation", "result": "observation",
}


def canonical_role(item: dict[str, Any], default: str = "observation") -> str:
    for key in ("role", "type", "speaker"):
        value = str(item.get(key, "")).lower()
        if value in ROLE_MAP:
            return ROLE_MAP[value]
    return default


def text_value(value: Any, limit: int = MAX_TEXT) -> str:
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, indent=2)
    if limit and len(text) > limit:
        return text[:limit] + f"... [recortado; {len(text)} caracteres]"
    return text


def state_snapshot(event: dict[str, Any]) -> dict[str, Any]:
    keys = ("state", "memory", "env", "score", "done", "action", "observation", "room", "inventory", "opened")
    return {key: event[key] for key in keys if key in event}


def normalize_events(record: dict[str, Any]) -> list[dict[str, Any]]:
    events: list[dict[str, Any]] = []
    for message in record.get("messages") or []:
        if isinstance(message, dict):
            events.append({"role": canonical_role(message, "assistant"), **message})
    for event in record.get("trace") or []:
        if isinstance(event, dict):
            item = dict(event)
            item["role"] = canonical_role(item)
            events.append(item)
    return events


In [81]:
def simple_diff(previous: dict[str, Any], current: dict[str, Any]) -> dict[str, Any]:
    keys = set(previous) | set(current)
    return {
        key: {"antes": previous.get(key), "despues": current.get(key)}
        for key in sorted(keys)
        if previous.get(key) != current.get(key)
    }


def print_state(event: dict[str, Any], previous_state: dict[str, Any]) -> dict[str, Any]:
    snapshot = state_snapshot(event)
    if not snapshot:
        return previous_state
    print("Estado relevante:")
    print(text_value(snapshot))
    diff = simple_diff(previous_state, snapshot) if previous_state else snapshot
    if diff:
        print("Cambio desde el paso anterior:")
        print(text_value(diff))
    return snapshot


def pretty_print_trace(record: dict[str, Any], max_text: int = MAX_TEXT) -> None:
    episode = " / ".join(str(record.get(key)) for key in ("scenario", "config", "repeat") if record.get(key) is not None)
    print("=" * 88)
    print(f"EPISODIO: {episode or '(sin identificador)'}")
    print("=" * 88)
    fields = ("status", "error", "goal", "goal_achieved", "goal_reason", "agent_error",
              "n_llm_calls", "n_tool_calls", "n_tool_errors", "tool_error_rate",
              "input_tokens", "output_tokens", "repaired_tool_calls", "goal_gate_triggers")
    for key in fields:
        if key in record and record[key] is not None:
            print(f"{key}: {text_value(record[key], max_text)}")

    prompt = record.get("initial_prompt") or record.get("system_prompt") or record.get("prompt")
    print("\n--- PROMPT / PEDIDO INICIAL ---")
    print(text_value(prompt, max_text) if prompt else "No persistido en este JSONL.")

    previous_state: dict[str, Any] = {}
    messages = record.get("messages") or []
    for number, message in enumerate(messages, 1):
        if not isinstance(message, dict):
            continue
        role = canonical_role(message, "assistant")
        print(f"\n--- MENSAJE {number} | rol={role} ---")
        print(f"Contenido: {text_value(message.get('content', message.get('text', '')), max_text)}")
        previous_state = print_state(message, previous_state)

    for event_number, event in enumerate(record.get("trace") or [], 1):
        if not isinstance(event, dict):
            continue
        tool = event.get("tool") or event.get("name", "(desconocida)")
        args = event.get("args", event.get("arguments", {}))
        timestamp = event.get("timestamp") or event.get("elapsed_s") or event.get("duration_s")
        time_label = f" | tiempo={timestamp}" if timestamp is not None else ""
        print(f"\n--- PASO {event_number}a | rol=assistant/tool_call{time_label} ---")
        # print("Nota: llamada reconstruida desde trace; el mensaje assistant crudo no fue persistido.")
        print(f"Herramienta solicitada: {tool}")
        print(f"Argumentos: {text_value(args, max_text)}")
        print(f"\n--- PASO {event_number}b | rol=tool/observation{time_label} ---")
        print(f"Herramienta ejecutada: {tool}")
        print(f"Output observado: {text_value(event.get('output', event.get('content', '')), max_text)}")
        if event.get("is_error") or event.get("exception"):
            print(f"Error: {event.get('exception') or event.get('is_error')}")
        previous_state = print_state(event, previous_state)

    if not messages and not record.get("trace"):
        print("\n(No hay mensajes ni eventos de trace persistidos.)")
    print("\n--- ESTADO FINAL / RESPUESTA ---")
    for key in ("final_state", "answer"):
        if key in record and record[key] is not None:
            print(f"{key}: {text_value(record[key], max_text)}")

## 6 y 7. Ejecutar un caso, procesar un lote y exportar

`CASE_INDEX` selecciona un episodio dentro del archivo cargado. También se puede procesar el lote completo y guardar una salida `.md` por archivo de entrada. Los contadores distinguen registros válidos, omitidos y errores de parseo.

In [82]:
def list_cases(records: list[dict[str, Any]]) -> None:
    for index, record in enumerate(records):
        print(f"[{index}] {record.get('scenario', '?')} | config={record.get('config', '?')} | rep={record.get('repeat', '?')} | status={record.get('status', '?')} | goal={record.get('goal_achieved', '?')}")


def process_file(path: str | Path, output_path: str | Path | None = None) -> dict[str, int]:
    meta, loaded, errors = load_jsonl(path)
    chunks = []
    for record in loaded:
        from io import StringIO
        import contextlib
        buffer = StringIO()
        with contextlib.redirect_stdout(buffer):
            pretty_print_trace(record)
        chunks.append(buffer.getvalue())
    text = "\n".join(chunks)
    if output_path:
        Path(output_path).write_text(text, encoding="utf-8")
        print(f"Salida guardada en {output_path}")
    return {"registros": len(loaded), "errores": len(errors), "omitidos": 0, "metadatos": len(meta)}

print("Casos del archivo seleccionado:")
list_cases(records)
if records:
    pretty_print_trace(records[0])

# Para procesar un lote, descomentá estas líneas:
# for path in files:
#     output = path.with_suffix(".trace.md")
#     print(path.name, process_file(path, output))


Casos del archivo seleccionado:
[0] study-with-key | config=baseline | rep=0 | status=ok | goal=True
[1] study-with-key | config=baseline | rep=1 | status=ok | goal=True
[2] study-with-key | config=baseline | rep=2 | status=ok | goal=True
[3] apartment-keys | config=baseline | rep=0 | status=ok | goal=True
[4] apartment-keys | config=baseline | rep=1 | status=ok | goal=False
[5] apartment-keys | config=baseline | rep=2 | status=ok | goal=False
[6] color-locks | config=baseline | rep=0 | status=ok | goal=False
[7] color-locks | config=baseline | rep=1 | status=ok | goal=True
[8] color-locks | config=baseline | rep=2 | status=ok | goal=True
[9] study-with-key | config=prompt_baseline | rep=0 | status=ok | goal=True
[10] study-with-key | config=prompt_baseline | rep=1 | status=ok | goal=True
[11] study-with-key | config=prompt_baseline | rep=2 | status=ok | goal=True
[12] apartment-keys | config=prompt_baseline | rep=0 | status=ok | goal=True
[13] apartment-keys | config=prompt_baseline |